# RoBERTa-large vs. TAPT RoBERTa-large — configurable seed

This notebook runs a controlled downstream comparison between the original `FacebookAI/roberta-large` encoder and the task-adapted checkpoint for `TAPT_COMPARISON_SEED`. Both models use the same seed, split, first-token pooling, classifier hyperparameters, loss, scheduler, and early-stopping rule. The only intended difference is the starting encoder checkpoint. The default seed is 13; set the environment variable to run another seed.

Results are written to `Code/BERT_models/experiments/tapt/downstream_comparison`. Existing results from the earlier random-pooler architecture are not reused.

In [ ]:
from pathlib import Path
import json
import os

PROJECT_DIR = next(
    (root for root in (Path.cwd(), *Path.cwd().parents)
     if (root / 'data' / 'annotate-lime.csv').exists()),
    None,
)
if PROJECT_DIR is None:
    raise FileNotFoundError('Could not locate the project root containing data/annotate-lime.csv.')
SOURCE_NOTEBOOK = PROJECT_DIR / 'Code' / 'BERT_models' / 'notebooks' / 'training' / 'BERT_Soft_Label_LR_Epoch_Tuning.ipynb'

source_notebook = json.loads(SOURCE_NOTEBOOK.read_text())
loaded_cells = 0
for cell_index, cell in enumerate(source_notebook['cells'], start=1):
    if cell.get('cell_type') != 'code':
        continue
    source = ''.join(cell.get('source', []))
    if source.lstrip().startswith('SMOKE_TEST = False'):
        break
    exec(compile(source, f'{SOURCE_NOTEBOOK.name}:cell-{cell_index}', 'exec'), globals())
    loaded_cells += 1

print(f'Loaded {loaded_cells} setup/definition cells from {SOURCE_NOTEBOOK.name}.')

In [ ]:
def environment_flag(name: str, default: bool) -> bool:
    raw_value = os.environ.get(name)
    if raw_value is None:
        return default
    normalized = raw_value.strip().lower()
    if normalized in {'1', 'true', 'yes', 'on'}:
        return True
    if normalized in {'0', 'false', 'no', 'off'}:
        return False
    raise ValueError(f'{name} must be a boolean value, got {raw_value!r}.')


COMPARISON_SEED = int(os.environ.get('TAPT_COMPARISON_SEED', '13'))
COMPARISON_OUTPUT_DIR = PROJECT_DIR / 'Code' / 'BERT_models' / 'experiments' / 'tapt' / 'downstream_comparison'
TAPT_CHECKPOINT_DIR = (
    PROJECT_DIR
    / 'Code'
    / 'BERT_models'
    / 'experiments'
    / 'tapt'
    / 'masked_language_modeling'
    / f'roberta_large_tapt_random_seed{COMPARISON_SEED}'
)

baseline_config = with_tuning_settings(
    ROBERTA_LARGE_CONFIG,
    learning_rate=1e-5,
    num_epochs=30,
    patience=6,
    scheduler_type='linear',
)

tapt_config = with_tuning_settings(
    make_roberta_large_tapt_config(TAPT_CHECKPOINT_DIR),
    learning_rate=1e-5,
    num_epochs=30,
    patience=6,
    scheduler_type='linear',
)

comparison_configs = [baseline_config, tapt_config]
COMPARISON_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# Fail before loading either large model if the TAPT checkpoint is incomplete or mismatched.
comparison_train_df, _, _ = make_random_split(dataset, COMPARISON_SEED, split_scheme='majority')
validate_tapt_checkpoint_for_run(tapt_config, f'random_seed{COMPARISON_SEED}', comparison_train_df)

display(pd.DataFrame([asdict(config) for config in comparison_configs]))
print('Comparison seed:', COMPARISON_SEED)
print('Comparison output directory:', COMPARISON_OUTPUT_DIR)
print('Validated TAPT checkpoint:', TAPT_CHECKPOINT_DIR)

In [ ]:
# Safe defaults: Run All reads completed results without retraining classifiers.
RUN_TAPT_COMPARISON = environment_flag('RUN_TAPT_COMPARISON', False)
SKIP_COMPLETED_COMPARISON = environment_flag('SKIP_COMPLETED_COMPARISON', True)
COMPARISON_STEM = f'roberta_large_tapt_comparison_seed{COMPARISON_SEED}'
COMPARISON_SUMMARY_PATH = COMPARISON_OUTPUT_DIR / f'{COMPARISON_STEM}.csv'

if RUN_TAPT_COMPARISON:
    comparison_summary = run_random_split_suite(
        model_configs=comparison_configs,
        seeds=[COMPARISON_SEED],
        loss_mode='binomial_nll',
        output_dir=COMPARISON_OUTPUT_DIR,
        summary_name=COMPARISON_STEM,
        skip_completed=SKIP_COMPLETED_COMPARISON,
    )
else:
    comparison_summary = (
        pd.read_csv(COMPARISON_SUMMARY_PATH)
        if COMPARISON_SUMMARY_PATH.exists()
        else pd.DataFrame()
    )
    print('Comparison training is disabled. Set RUN_TAPT_COMPARISON = True to opt in.')
    if not comparison_summary.empty:
        print('Loaded existing comparison:', COMPARISON_SUMMARY_PATH)

comparison_columns = [
    'experiment_name',
    'model_name',
    'run_id',
    'pooling',
    'best_epoch',
    'best_val_loss',
    'macro_mae',
    'mean_pearson',
    'majority_tuned_macro_f1',
]
if comparison_summary.empty:
    comparison_table = pd.DataFrame(columns=comparison_columns)
    print('No completed comparison results were found.')
else:
    comparison_table = comparison_summary[comparison_columns].sort_values('experiment_name')
    display(comparison_table)
    if RUN_TAPT_COMPARISON:
        comparison_table.to_csv(
            COMPARISON_OUTPUT_DIR / f'{COMPARISON_STEM}_key_metrics.csv',
            index=False,
        )

## Interpretation

For the primary soft-label comparison, lower `best_val_loss` and `macro_mae` are better, while higher `mean_pearson` is better. `majority_tuned_macro_f1` is a secondary thresholded metric. Treat each run as a controlled single-seed comparison; combine matched seeds before making a general claim about TAPT.

## Final results summary

For the selected seed, this notebook compares standard `FacebookAI/roberta-large` with the matching TAPT encoder. Both models use the same 424/91/91 train-validation-test split, first-token pooling, batch size 4, learning rate `1e-5`, linear scheduler, maximum of 30 epochs, and early-stopping patience of 6. The only intended difference is whether the encoder first underwent task-adaptive masked-language-model training.

Use the generated comparison table and CSV files to interpret the selected seed. Compare the direction and size of the standard-versus-TAPT differences across matched seeds before drawing a general conclusion. Leave-one-situation-out evaluation would be needed to assess transfer to completely unseen situations.